In [0]:
%run ./ingest_common_helpers

In [0]:
%run ./ingest_manifest_writer

In [0]:
# Remotive API Configuration
REMOTIVE_API = "https://remotive.com/api/remote-jobs"
SOURCE_NAME = "remotive"

# Remotive-specific field mapping
REMOTIVE_FIELDS = {
    'job_id': 'id',                              # int64 - numeric job ID
    'url': 'url',                                # string
    'title': 'title',                            # string
    'company': 'company_name',                   # string
    'company_logo': 'company_logo',              # string
    'category': 'category',                      # string
    'tags': 'tags',                              # array/object
    'job_type': 'job_type',                      # string (singular)
    'publication_date': 'publication_date',      # string - ISO date
    'location': 'candidate_required_location',   # string
    'salary': 'salary',                          # string
    'description': 'description',                # string
    'company_logo_url': 'company_logo_url'       # string
}

In [0]:
def extract_remotive_job_id(job):
    """Extract job ID from Remotive record"""
    return str(job.get('id', ''))

def validate_remotive_record(job):
    """Validate Remotive job record structure"""
    required_fields = ['id', 'title', 'company_name', 'url']
    for field in required_fields:
        if not job.get(field):
            return False, f"Missing required field: {field}"
    
    # Validate data types
    if not isinstance(job.get('id'), int):
        return False, "id must be int64"
    
    # Validate publication_date format (if present)
    pub_date = job.get('publication_date')
    if pub_date and not isinstance(pub_date, str):
        return False, "publication_date must be string (ISO date format)"
    
    return True, None

def fetch_remotive_jobs():
    """Fetch jobs from Remotive API"""
    try:
        response = requests.get(REMOTIVE_API, timeout=30)
        response.raise_for_status()
        data = response.json()
        jobs = data.get('jobs', [])
        
        # Validate records
        validated_jobs = []
        for job in jobs:
            is_valid, error = validate_remotive_record(job)
            if is_valid:
                validated_jobs.append(job)
            else:
                print(f"  WARNING: Skipping invalid record - {error}")
        
        return validated_jobs, None
    except Exception as e:
        return None, str(e)

In [0]:
print("LMIP BRONZE LAYER INGESTION - REMOTIVE")
print("Started:", datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'))
print("Target:", BRONZE_JOB_SNAPSHOT)

# Execute ingestion
remotive_success = ingest_jobs(
    source_name=SOURCE_NAME,
    fetch_function=fetch_remotive_jobs,
    api_url=REMOTIVE_API,
    extract_job_id_func=extract_remotive_job_id,
    log_api_response_func=log_api_response,
    start_pipeline_run_func=start_pipeline_run,
    complete_pipeline_run_func=complete_pipeline_run,
    log_audit_func=log_audit_pipeline_run
)

print(f"Status: {'SUCCESS' if remotive_success else 'FAILED'}")
print(f"Completed: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")


In [0]:
%sql
-- View recent Remotive bronze job snapshots
SELECT 
  source_name,
  batch_id,
  COUNT(*) as record_count,
  MIN(ingestion_timestamp) as first_ingested,
  MAX(ingestion_timestamp) as last_ingested
FROM bronze.bronze_job_snapshot
WHERE source_name = 'remotive'
GROUP BY source_name, batch_id
ORDER BY last_ingested DESC
LIMIT 10

In [0]:
%sql
-- View Remotive pipeline audit history
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  start_time,
  end_time,
  rows_read,
  rows_written,
  ROUND(runtime_seconds, 2) as runtime_sec,
  error_message
FROM audit.audit_pipeline_runs
WHERE pipeline_name = 'bronze_ingestion_remotive'
ORDER BY start_time DESC
LIMIT 20

In [0]:
%sql
-- View Remotive API response telemetry
SELECT 
  source_name,
  batch_id,
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit,
  logged_at
FROM bronze.bronze_api_response_log
WHERE source_name = 'remotive'
ORDER BY logged_at DESC
LIMIT 20